In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

# Set seed
torch.manual_seed(42)

# Verify RLE Encoding/Decoding Functions
def rle_encode(mask):
    """mask: a 384x384 boolean/uint8 array -> RLE string (Fortran order)."""
    px = np.asarray(mask, np.uint8).T.flatten()
    px = np.concatenate([[0], px, [0]])
    runs = np.where(px[1:] != px[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(int(x)) for x in runs)

def rle_decode(rle, size=384):
    """RLE string -> 384x384 uint8 array."""
    m = np.zeros(size * size, np.uint8)
    if not isinstance(rle, str) or not rle.strip():
        return m.reshape(size, size, order="F")
    a = np.asarray(rle.split(), np.int64)
    for s, l in zip(a[0::2], a[1::2]):
        m[s - 1: s - 1 + l] = 1
    return m.reshape(size, size, order="F")

# Quick Sanity Check
sample_mask = (np.random.rand(384, 384) > 0.8).astype(np.uint8)
assert np.array_equal(rle_decode(rle_encode(sample_mask)), sample_mask)
print("✅ RLE sanity check passed!")

✅ RLE sanity check passed!


In [5]:
class PartsDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, is_test=False):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*.png")))
        self.mask_dir = mask_dir
        self.is_test = is_test

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = Image.open(img_path).convert("RGB")
        img_arr = np.array(img, dtype=np.float32) / 255.0  # Normalize to [0, 1]
        img_tensor = torch.tensor(img_arr).permute(2, 0, 1) # [C, H, W]

        if not self.is_test:
            # DO NOT use .convert("RGB") on masks to retain raw class IDs (0-6)
            mask_path = os.path.join(self.mask_dir, os.path.basename(img_path))
            mask = np.array(Image.open(mask_path), dtype=np.int64) # Class IDs 0-6
            mask_tensor = torch.tensor(mask, dtype=torch.long)
            return img_tensor, mask_tensor
        
        return img_tensor, os.path.basename(img_path)

# Custom Lightweight U-Net Architecture (7 classes: 0=background, 1-6=parts)
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.enc1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        
        self.bottleneck = DoubleConv(128, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)
        
        self.final = nn.Conv2d(32, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final(d1)

## STEP 3

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Updated paths based on your Kaggle directory output
BASE_DIR = "/kaggle/input/competitions/nppe-1-dl-gen-ai-course-semantic-segmentation/Dataset"
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "train/images")
TRAIN_MASK_DIR = os.path.join(BASE_DIR, "train/masks")

# Build dataset and dataloader
train_dataset = PartsDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR)
print(f"Loaded {len(train_dataset)} training samples.")  # Should show 2000

train_loader = DataLoader(
    train_dataset, 
    batch_size=16, 
    shuffle=True, 
    num_workers=2, 
    pin_memory=True
)

# Initialize Model, Loss, Optimizer, and Mixed Precision Scaler
model = UNet(num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

# Fast 8-epoch training loop
EPOCHS = 8
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * imgs.size(0)
        
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f}")

Using device: cuda
Loaded 2000 training samples.


/tmp/ipykernel_58/4099156869.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_58/4099156869.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [1/8] - Loss: 0.9020
Epoch [2/8] - Loss: 0.3440
Epoch [3/8] - Loss: 0.1920
Epoch [4/8] - Loss: 0.1206
Epoch [5/8] - Loss: 0.0857
Epoch [6/8] - Loss: 0.0762
Epoch [7/8] - Loss: 0.0594
Epoch [8/8] - Loss: 0.0519


## step 4

In [9]:
# # Load sample_submission.csv to guarantee exact row order and ID alignment
# sample_sub_path = os.path.join(BASE_DIR, "sample_submission.csv")
# sample_sub = pd.read_csv(sample_sub_path)

# model.eval()
# predictions_dict = {}

# # 1. Run inference on all test images
# with torch.no_grad():
#     for imgs, filenames in tqdm(test_loader, desc="Generating Predictions"):
#         imgs = imgs.to(device)
#         with torch.amp.autocast('cuda'):
#             preds = torch.argmax(model(imgs), dim=1).cpu().numpy()  # [B, H, W]
        
#         for pred, fname in zip(preds, filenames):
#             img_id = fname.replace(".png", "")
#             for class_id in range(1, 7):
#                 binary_mask = (pred == class_id).astype(np.uint8)
#                 rle_str = rle_encode(binary_mask)
#                 predictions_dict[f"{img_id}_{class_id}"] = rle_str if rle_str else "1 1"

# # 2. Map predictions strictly matching sample_submission.csv order
# submission_rows = []
# for row_id in sample_sub["ImageId_ClassId"]:
#     rle_val = predictions_dict.get(row_id, "1 1")
#     submission_rows.append({
#         "ImageId_ClassId": row_id,
#         "EncodedPixels": rle_val
#     })

# sub_df = pd.DataFrame(submission_rows)
# sub_df.to_csv("submission.csv", index=False)

# print("✅ Fixed submission.csv successfully generated!")
# print(f"Total Rows: {len(sub_df)} (Expected: {len(sample_sub)})")
# print(sub_df.head(10))

Generating Predictions: 100%|██████████| 63/63 [00:08<00:00,  7.76it/s]


✅ Fixed submission.csv successfully generated!
Total Rows: 3000 (Expected: 3000)
              ImageId_ClassId EncodedPixels
0       img_00001.png_hex_nut           1 1
1        img_00001.png_washer           1 1
2          img_00001.png_bolt           1 1
3  img_00001.png_ball_bearing           1 1
4        img_00001.png_spring           1 1
5        img_00001.png_o_ring           1 1
6       img_00005.png_hex_nut           1 1
7        img_00005.png_washer           1 1
8          img_00005.png_bolt           1 1
9  img_00005.png_ball_bearing           1 1


In [15]:
# Class ID to Class Name Mapping
CLASS_MAP = {
    1: "hex_nut",
    2: "washer",
    3: "bolt",
    4: "ball_bearing",
    5: "spring",
    6: "o_ring"
}

# 1. Re-initialize Test Dataset and DataLoader
TEST_IMG_DIR = os.path.join(BASE_DIR, "test/images")
test_dataset = PartsDataset(TEST_IMG_DIR, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
print(f"Loaded {len(test_dataset)} test samples.")

# 2. Load sample_submission.csv for exact ID ordering
sample_sub_path = os.path.join(BASE_DIR, "sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path)

# 3. Choose active model (use 'enhanced_model' or 'model')
active_model = enhanced_model if 'enhanced_model' in globals() else model
active_model.eval()

predictions_dict = {}

# 4. Generate Predictions with correct Class Names
with torch.no_grad():
    for imgs, filenames in tqdm(test_loader, desc="Generating Predictions"):
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda'):
            preds = torch.argmax(active_model(imgs), dim=1).cpu().numpy()  # [B, H, W]
        
        for pred, fname in zip(preds, filenames):
            for class_id, class_name in CLASS_MAP.items():
                binary_mask = (pred == class_id).astype(np.uint8)
                rle_str = rle_encode(binary_mask)
                
                # Format: img_00001.png_hex_nut
                row_key = f"{fname}_{class_name}"
                predictions_dict[row_key] = rle_str if rle_str else "1 1"

# 5. Map predictions strictly matching sample_submission.csv order
submission_rows = []
for row_id in sample_sub["ImageId_ClassId"]:
    rle_val = predictions_dict.get(row_id, "1 1")
    submission_rows.append({
        "ImageId_ClassId": row_id,
        "EncodedPixels": rle_val
    })

sub_df = pd.DataFrame(submission_rows)
sub_df.to_csv("submission.csv", index=False)

print("✅ submission.csv successfully generated with actual RLE predictions!")
print(f"Total Rows: {len(sub_df)} (Matches sample_submission: {len(sub_df) == len(sample_sub)})")
print(sub_df.head(10))

Loaded 500 test samples.


Generating Predictions: 100%|██████████| 63/63 [00:08<00:00,  7.57it/s]


✅ submission.csv successfully generated with actual RLE predictions!
Total Rows: 3000 (Matches sample_submission: True)
              ImageId_ClassId  \
0       img_00001.png_hex_nut   
1        img_00001.png_washer   
2          img_00001.png_bolt   
3  img_00001.png_ball_bearing   
4        img_00001.png_spring   
5        img_00001.png_o_ring   
6       img_00005.png_hex_nut   
7        img_00005.png_washer   
8          img_00005.png_bolt   
9  img_00005.png_ball_bearing   

                                       EncodedPixels  
0  17069 6 17451 10 17833 14 18215 18 18597 22 18...  
1  20 9 402 12 678 13 786 12 1063 11 1169 13 1445...  
2  32839 1 33222 3 33604 6 33987 13 34371 16 3475...  
3  20659 13 20675 1 21040 24 21423 27 21805 31 22...  
4  29 21 414 21 798 21 1182 22 1567 21 1951 22 23...  
5  19509 3 19892 5 20276 15 20294 1 20672 3 22952...  
6  75477 7 75848 1 75850 19 76221 33 76599 40 769...  
7  36 27 80 27 420 27 464 28 803 29 848 29 1187 2...  
8  32843 2 33225 6 33

In [19]:
import os
import gc
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ---------------------------------------------------------
# 0. Clean GPU Memory & Environment Setup
# ---------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

BASE_DIR = "/kaggle/input/competitions/nppe-1-dl-gen-ai-course-semantic-segmentation/Dataset"
TRAIN_IMG_DIR = os.path.join(BASE_DIR, "train/images")
TRAIN_MASK_DIR = os.path.join(BASE_DIR, "train/masks")
TEST_IMG_DIR = os.path.join(BASE_DIR, "test/images")

CLASS_MAP = {
    1: "hex_nut",
    2: "washer",
    3: "bolt",
    4: "ball_bearing",
    5: "spring",
    6: "o_ring"
}

# ---------------------------------------------------------
# 1. Dataset Class with Albumentations & Correct Casting
# ---------------------------------------------------------
class EnhancedPartsDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, transform=None, is_test=False):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*.png")))
        self.mask_dir = mask_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = np.array(Image.open(img_path).convert("RGB"))

        if not self.is_test:
            mask_path = os.path.join(self.mask_dir, os.path.basename(img_path))
            mask = np.array(Image.open(mask_path), dtype=np.int64)

            if self.transform:
                augmented = self.transform(image=img, mask=mask)
                img = augmented['image']
                mask = torch.tensor(augmented['mask'], dtype=torch.long) # Fixed Long Tensor
            else:
                img = torch.tensor(img / 255.0, dtype=torch.float32).permute(2, 0, 1)
                mask = torch.tensor(mask, dtype=torch.long)
                
            return img, mask
        
        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']
        else:
            img = torch.tensor(img / 255.0, dtype=torch.float32).permute(2, 0, 1)

        return img, os.path.basename(img_path)

# Data Transforms
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=30, p=0.5, border_mode=0),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

test_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# ---------------------------------------------------------
# 2. Focal + Soft Dice Combo Loss
# ---------------------------------------------------------
class FocalDiceLoss(nn.Module):
    def __init__(self, num_classes=7, gamma=2.0, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma
        self.smooth = smooth
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()

        probs = torch.softmax(inputs, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes=self.num_classes).permute(0, 3, 1, 2).float()
        
        intersection = torch.sum(probs * targets_one_hot, dim=(0, 2, 3))
        cardinality = torch.sum(probs + targets_one_hot, dim=(0, 2, 3))
        
        dice_score = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        dice_loss = 1.0 - torch.mean(dice_score[1:]) # Class 1 to 6 focus
        
        return focal_loss + dice_loss

# ---------------------------------------------------------
# 3. Enhanced U-Net with Correct Decoder Dimensions
# ---------------------------------------------------------
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class EnhancedUNet(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.enc1 = DoubleConv(3, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        
        self.bottleneck = DoubleConv(256, 512)
        
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)  # Fixed Channel Math
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)   # Fixed Channel Math
        
        self.final = nn.Conv2d(64, num_classes, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final(d1)

# ---------------------------------------------------------
# 4. Training Loop (VRAM-Optimized with Gradient Accumulation)
# ---------------------------------------------------------
train_dataset = EnhancedPartsDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)

enhanced_model = EnhancedUNet(num_classes=7).to(device)
criterion = FocalDiceLoss(num_classes=7)
optimizer = optim.AdamW(enhanced_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

ACCUMULATION_STEPS = 2
EPOCHS = 15

print("🔥 Starting Training Loop...")
enhanced_model.train()
optimizer.zero_grad()

for epoch in range(EPOCHS):
    running_loss = 0.0
    
    for i, (imgs, masks) in enumerate(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        
        with torch.amp.autocast('cuda'):
            outputs = enhanced_model(imgs)
            loss = criterion(outputs, masks)
            loss = loss / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * ACCUMULATION_STEPS * imgs.size(0)

    scheduler.step()
    epoch_loss = running_loss / len(train_dataset)
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch [{epoch+1:02d}/{EPOCHS:02d}] - Loss: {epoch_loss:.4f} | LR: {current_lr:.6f}")

torch.save(enhanced_model.state_dict(), "enhanced_unet.pth")
print("✅ Training Complete & Model Weights Saved!")

# ---------------------------------------------------------
# 5. Inference & Submission File Generation
# ---------------------------------------------------------
test_dataset = EnhancedPartsDataset(TEST_IMG_DIR, transform=test_transform, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

sample_sub_path = os.path.join(BASE_DIR, "sample_submission.csv")
sample_sub = pd.read_csv(sample_sub_path)

enhanced_model.eval()
predictions_dict = {}

with torch.no_grad():
    for imgs, filenames in tqdm(test_loader, desc="Generating Predictions"):
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda'):
            preds = torch.argmax(enhanced_model(imgs), dim=1).cpu().numpy()
        
        for pred, fname in zip(preds, filenames):
            for class_id, class_name in CLASS_MAP.items():
                binary_mask = (pred == class_id).astype(np.uint8)
                rle_str = rle_encode(binary_mask)
                
                row_key = f"{fname}_{class_name}"
                predictions_dict[row_key] = rle_str if rle_str else "1 1"

submission_rows = []
for row_id in sample_sub["ImageId_ClassId"]:
    rle_val = predictions_dict.get(row_id, "1 1")
    submission_rows.append({
        "ImageId_ClassId": row_id,
        "EncodedPixels": rle_val
    })

sub_df = pd.DataFrame(submission_rows)
sub_df.to_csv("submission.csv", index=False)

print("✅ Enhanced submission.csv generated successfully!")
print(f"Matches sample_submission length: {len(sub_df) == len(sample_sub)}")
print(sub_df.head(10))

🚀 Using device: cuda
🔥 Starting Training Loop...


/tmp/ipykernel_58/2651521612.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(augmented['mask'], dtype=torch.long) # Fixed Long Tensor
/tmp/ipykernel_58/2651521612.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(augmented['mask'], dtype=torch.long) # Fixed Long Tensor


Epoch [01/15] - Loss: 1.3479 | LR: 0.000989


/tmp/ipykernel_58/2651521612.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(augmented['mask'], dtype=torch.long) # Fixed Long Tensor
/tmp/ipykernel_58/2651521612.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(augmented['mask'], dtype=torch.long) # Fixed Long Tensor


Epoch [02/15] - Loss: 0.9386 | LR: 0.000957


/tmp/ipykernel_58/2651521612.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(augmented['mask'], dtype=torch.long) # Fixed Long Tensor
/tmp/ipykernel_58/2651521612.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(augmented['mask'], dtype=torch.long) # Fixed Long Tensor


... (23 outputs left out)

/tmp/ipykernel_58/2651521612.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(augmented['mask'], dtype=torch.long) # Fixed Long Tensor
/tmp/ipykernel_58/2651521612.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(augmented['mask'], dtype=torch.long) # Fixed Long Tensor


Epoch [15/15] - Loss: 0.0697 | LR: 0.000001
✅ Training Complete & Model Weights Saved!


Generating Predictions: 100%|██████████| 63/63 [00:13<00:00,  4.78it/s]


✅ Enhanced submission.csv generated successfully!
Matches sample_submission length: True
              ImageId_ClassId  \
0       img_00001.png_hex_nut   
1        img_00001.png_washer   
2          img_00001.png_bolt   
3  img_00001.png_ball_bearing   
4        img_00001.png_spring   
5        img_00001.png_o_ring   
6       img_00005.png_hex_nut   
7        img_00005.png_washer   
8          img_00005.png_bolt   
9  img_00005.png_ball_bearing   

                                       EncodedPixels  
0  17070 3 17452 8 17833 14 17848 1 18215 18 1859...  
1  1837 2 2210 16 2591 23 2973 29 3098 1 3354 34 ...  
2  32064 2 32437 18 32821 20 33203 27 33585 30 33...  
3  20662 1 20664 3 20668 1 20670 5 21042 22 21424...  
4  23 26 403 32 787 32 1172 31 1552 36 1934 39 23...  
5  73503 11 73884 19 74265 25 74647 30 75028 35 7...  
6  75477 8 75848 21 76221 33 76599 40 76982 42 77...  
7  36 33 77 1 79 28 420 29 450 3 456 3 460 1 462 ...  
8  33225 5 33608 9 33990 15 34374 19 34756 25 351...

# to not train unnecessarily 

In [ ]:
MODEL_PATH = "unet_model.pth"

# Train only if saved model weights don't exist yet
if not os.path.exists(MODEL_PATH):
    print("No saved weights found. Starting training...")
    model.train()
    for epoch in range(EPOCHS):
        running_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
                loss = criterion(outputs, masks)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item() * imgs.size(0)
            
        epoch_loss = running_loss / len(train_dataset)
        print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {epoch_loss:.4f}")
        
    # Save trained weights
    torch.save(model.state_dict(), MODEL_PATH)
    print(f"✅ Model saved to {MODEL_PATH}")
else:
    print(f"✅ Found existing weights at {MODEL_PATH}. Skipping training!")
    model.load_state_dict(torch.load(MODEL_PATH))